# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

In [ ]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty



class DataProcessor:
    def __init__(self):
        self._queue = ArrayQueue()
        self._history = ArrayStack()
        self._state = []

    def add(self, record):
        """Agrega un registro a la cola de pendientes. No lo procesa."""
        try:
            n = len(record)
        except TypeError:
            raise DataProcessor.InvalidRecord(
                "El registro debe ser una tupla (sensor, variable, value)."
            )
        if n != 3:
            raise DataProcessor.InvalidRecord(
                "El registro debe tener exactamente 3 componentes."
            )
        sensor, variable, value = record
        if isinstance(value, bool) or not isinstance(value, numbers.Number):
            raise DataProcessor.InvalidRecord("El 'value' del registro debe ser numérico.")
        self._queue.enqueue((sensor, variable, value))

    def _find_index(self, sensor, variable):
        """Busca (sensor, variable) en el estado actual. -1 si no existe. O(n)."""
        for i in range(len(self._state)):
            s, v, _ = self._state[i]
            if s == sensor and v == variable:
                return i
        return -1

    def process_next(self):
        """Procesa el registro más antiguo de la cola y actualiza el estado."""
        record = self._queue.dequeue()  # Empty si la cola está vacía
        sensor, variable, value = record

        idx = self._find_index(sensor, variable)
        if idx == -1:
            previous_value = None
            existed_before = False
            self._state.append(record)
        else:
            previous_value = self._state[idx][2]
            existed_before = True
            self._state[idx] = record

        self._history.push((record, previous_value, existed_before))
        return record

    def undo(self):
        """Deshace el último cambio aplicado por process_next(). LIFO."""
        record, previous_value, existed_before = self._history.pop()  # Empty si vacío
        sensor, variable, _ = record

        idx = self._find_index(sensor, variable)
        if existed_before:
            self._state[idx] = (sensor, variable, previous_value)
        else:
            del self._state[idx]

    def pending(self):
        """Número de registros que aún esperan ser procesados."""
        return len(self._queue)